[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AltamarMx/anomalias-2026-2/blob/main/notebooks/019_stl_teoria.ipynb)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import STL
from sklearn.ensemble import IsolationForest
import pandas as pd

plt.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)
rng = np.random.default_rng(42)
from IPython.display import display, Markdown

# Semana 10A — Detección basada en Descomposición (Teoría)

**Objetivo:** Usar STL para descomponer la serie y buscar anomalías
en los residuales (ya limpios de tendencia y estacionalidad).
Introducir Isolation Forest como enfoque multivariado complementario.

**Temas:**

1. STL — ¿qué hace y por qué es mejor que `seasonal_decompose`?
2. Detección de outliers en residuales
3. Umbrales adaptativos
4. Isolation Forest (concepto)

---
## 1. STL — Seasonal-Trend decomposition using LOESS

STL descompone una serie en tres componentes:
$$y_t = T_t + S_t + R_t$$

**Ventajas sobre `seasonal_decompose`:**

| Característica | `seasonal_decompose` | STL |
|:---|:---|:---|
| Método de tendencia | Media móvil simple | LOESS (regresión local ponderada) |
| Robustez a outliers | Ninguna | Itera re-ponderando outliers |
| Estacionalidad | Fija (promedio por posición) | Puede variar lentamente |
| Bordes | Pierde `period/2` datos | Maneja mejor los extremos |

La clave: STL **pondera menos** las observaciones extremas en cada
iteración de LOESS, así los outliers no contaminan la estimación
de tendencia ni estacionalidad → quedan aislados en el residuo.

In [ ]:
# Señal sintética con estacionalidad + tendencia + ruido + outliers
n_pts = 730
t = np.arange(n_pts)
tendencia = 20 + 0.005 * t
estacionalidad = 8 * np.sin(2 * np.pi * t / 365)
ruido = rng.normal(0, 1, n_pts)
y_limpia = tendencia + estacionalidad + ruido

# Inyectar 5 outliers
idx_outliers = np.array([60, 200, 350, 500, 650])
y_contaminada = y_limpia.copy()
y_contaminada[idx_outliers] += rng.choice([-15, 15], len(idx_outliers))

serie = pd.Series(y_contaminada, index=pd.date_range("2023-01-01", periods=n_pts, freq="D"))

# Aplicar STL
stl_result = STL(serie, period=365, robust=True).fit()

fig1, axes1 = plt.subplots(4, 1, figsize=(13, 10), sharex=True)

axes1[0].plot(serie.index, serie.values, "steelblue", lw=0.8)
axes1[0].scatter(serie.index[idx_outliers], serie.values[idx_outliers],
                 c="crimson", s=60, zorder=5, label="Outliers inyectados")
axes1[0].set_ylabel("Observada")
axes1[0].set_title("Serie sintética con 5 outliers")
axes1[0].legend()

axes1[1].plot(serie.index, stl_result.trend, "crimson", lw=2)
axes1[1].set_ylabel("Tendencia")

axes1[2].plot(serie.index, stl_result.seasonal, "seagreen", lw=1)
axes1[2].set_ylabel("Estacionalidad")

axes1[3].plot(serie.index, stl_result.resid, "gray", lw=0.8)
axes1[3].scatter(serie.index[idx_outliers], stl_result.resid.values[idx_outliers],
                 c="crimson", s=60, zorder=5)
axes1[3].set_ylabel("Residuo")
axes1[3].set_xlabel("Fecha")
axes1[3].set_title("Los outliers aparecen como picos en el residuo")

plt.tight_layout()
plt.show()

> **Observa:** Los 5 outliers inyectados aparecen claramente como picos
> en el panel de residuos.  La tendencia y la estacionalidad no se
> contaminaron gracias a `robust=True`.
>
> Ahora podemos aplicar z-score, MAD o IQR **sobre los residuales**
> en vez de sobre la serie bruta — mucho más limpio.

---
## 2. Detección de outliers en residuales

Sobre los residuales de STL aplicamos los mismos métodos de la
semana 9, pero ahora sin la contaminación de tendencia y estacionalidad.

In [ ]:
resid = stl_result.resid.dropna()
resid_vals = resid.values

# Z-score sobre residuales
mu_r = np.mean(resid_vals)
sigma_r = np.std(resid_vals)
z_resid = (resid_vals - mu_r) / sigma_r

# MAD sobre residuales
med_r = np.median(resid_vals)
mad_r = np.median(np.abs(resid_vals - med_r))
mad_scaled = 1.4826 * mad_r

# Umbrales
umbral_z = 3 * sigma_r
umbral_mad = 3 * mad_scaled

out_z = np.abs(resid_vals - mu_r) > umbral_z
out_mad = np.abs(resid_vals - med_r) > umbral_mad

fig2, ax2 = plt.subplots(figsize=(13, 4))
ax2.plot(resid.index, resid_vals, "gray", lw=0.6)
ax2.axhline(mu_r + umbral_z, color="crimson", ls="--", lw=1.5,
            label=f"±3σ ({out_z.sum()} det.)")
ax2.axhline(mu_r - umbral_z, color="crimson", ls="--", lw=1.5)
ax2.axhline(med_r + umbral_mad, color="darkorange", ls=":", lw=1.5,
            label=f"±3·MAD ({out_mad.sum()} det.)")
ax2.axhline(med_r - umbral_mad, color="darkorange", ls=":", lw=1.5)

ax2.scatter(resid.index[out_mad], resid_vals[out_mad],
            c="crimson", s=40, zorder=5)

# Marcar outliers reales
ax2.scatter(resid.index[idx_outliers], resid_vals[idx_outliers],
            facecolors="none", edgecolors="black", s=100, lw=2, zorder=6,
            label="Outliers reales")

ax2.set_ylabel("Residuo")
ax2.set_xlabel("Fecha")
ax2.set_title("Detección sobre residuales de STL")
ax2.legend()
plt.tight_layout()
plt.show()

> Los 5 outliers reales (círculos negros) son detectados correctamente.
> No hay falsos positivos estacionales porque la estacionalidad
> ya fue removida por STL antes de aplicar los umbrales.

---
## 3. Umbrales adaptativos

El umbral ±3σ asume que la varianza de los residuales es **constante**.
¿Y si no lo es?

**Ejemplo:** Una señal cuya variabilidad cambia entre estaciones
(más ruido en verano, menos en invierno).

**Soluciones:**
- **Umbral por ventana:** calcular σ en ventanas de N días
- **Percentiles:** usar percentiles 0.5% y 99.5% (no asumen normalidad)

In [ ]:
# Señal con varianza cambiante
n_pts2 = 730
t2 = np.arange(n_pts2)
sigma_variable = 0.5 + 2.0 * (0.5 + 0.5 * np.sin(2 * np.pi * t2 / 365))
y2 = 20 + 5 * np.sin(2 * np.pi * t2 / 365) + sigma_variable * rng.normal(0, 1, n_pts2)

# Inyectar outliers en zona de baja varianza (invierno)
idx_out2 = np.array([30, 395])
y2[idx_out2] += np.array([6, 6])

serie2 = pd.Series(y2, index=pd.date_range("2023-01-01", periods=n_pts2, freq="D"))
stl2 = STL(serie2, period=365, robust=True).fit()
resid2 = stl2.resid.dropna().values

# Umbral fijo
sigma_global = np.std(resid2)
out_fijo = np.abs(resid2) > 3 * sigma_global

# Umbral por percentiles
p_lo, p_hi = np.percentile(resid2, [0.5, 99.5])
out_pct = (resid2 < p_lo) | (resid2 > p_hi)

fig3, (ax3a, ax3b) = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

# Fijo
ax3a.plot(serie2.index, resid2, "gray", lw=0.6)
ax3a.axhline(3 * sigma_global, color="crimson", ls="--", lw=1.5)
ax3a.axhline(-3 * sigma_global, color="crimson", ls="--", lw=1.5)
ax3a.scatter(serie2.index[out_fijo], resid2[out_fijo],
             c="crimson", s=30, zorder=5)
ax3a.scatter(serie2.index[idx_out2], resid2[idx_out2],
             facecolors="none", edgecolors="black", s=100, lw=2, zorder=6)
ax3a.set_ylabel("Residuo")
ax3a.set_title(f"Umbral fijo ±3σ — {out_fijo.sum()} detecciones")

# Percentiles
ax3b.plot(serie2.index, resid2, "gray", lw=0.6)
ax3b.axhline(p_hi, color="darkorange", ls="--", lw=1.5, label=f"P99.5 = {p_hi:.2f}")
ax3b.axhline(p_lo, color="darkorange", ls="--", lw=1.5, label=f"P0.5 = {p_lo:.2f}")
ax3b.scatter(serie2.index[out_pct], resid2[out_pct],
             c="darkorange", s=30, zorder=5)
ax3b.scatter(serie2.index[idx_out2], resid2[idx_out2],
             facecolors="none", edgecolors="black", s=100, lw=2, zorder=6)
ax3b.set_ylabel("Residuo")
ax3b.set_xlabel("Fecha")
ax3b.set_title(f"Umbral por percentiles (0.5%, 99.5%) — {out_pct.sum()} detecciones")
ax3b.legend()

plt.tight_layout()
plt.show()

> **Observa:** Cuando la varianza cambia en el tiempo, el umbral
> fijo ±3σ puede ser demasiado laxo en las zonas de baja varianza
> (no detecta outliers allí) y demasiado estricto en zonas de alta
> varianza (falsos positivos).
>
> Los percentiles son una alternativa simple que no asume normalidad
> ni varianza constante — pero tampoco se adaptan localmente.
> Una tercera opción (no implementada aquí) es calcular σ en ventanas
> deslizantes.

---
## 4. Isolation Forest — concepto

Cambio de paradigma: en vez de definir qué es "normal" y buscar lo
que no cabe, Isolation Forest pregunta:

> **¿Qué tan fácil es aislar esta observación?**

**Idea:** Construir árboles de decisión con cortes aleatorios.
- Los **outliers** están lejos del resto → se aíslan con pocos cortes.
- Los **datos normales** están en zonas densas → necesitan muchos cortes.

El *anomaly score* es proporcional a la profundidad promedio del árbol
necesaria para aislar cada punto. Puntos con poca profundidad = anomalías.

**Ventajas:**
- No asume distribución ni estacionalidad
- Naturalmente **multivariado** (puede usar múltiples variables a la vez)
- El parámetro `contamination` controla la fracción esperada de anomalías

In [ ]:
# Datos 2D: cluster normal + 5 anomalías
n_normal = 300
x_normal = rng.normal(0, 1, (n_normal, 2))
x_anomalias = np.array([
    [4, 3], [-3, 4], [3, -3], [-4, -2], [0, 5]
])
x_total = np.vstack([x_normal, x_anomalias])
etiquetas_reales = np.array([0] * n_normal + [1] * 5)

# Isolation Forest
iforest = IsolationForest(contamination=0.02, random_state=42)
predicciones = iforest.fit_predict(x_total)
scores = iforest.decision_function(x_total)

out_if = predicciones == -1

fig4, (ax4a, ax4b) = plt.subplots(1, 2, figsize=(12, 5))

# Scatter con predicciones
ax4a.scatter(x_total[~out_if, 0], x_total[~out_if, 1],
             c="steelblue", s=15, alpha=0.5, label="Normal")
ax4a.scatter(x_total[out_if, 0], x_total[out_if, 1],
             c="crimson", s=60, zorder=5, marker="X", label="Anomalía (IF)")
ax4a.scatter(x_anomalias[:, 0], x_anomalias[:, 1],
             facecolors="none", edgecolors="black", s=120, lw=2, zorder=6,
             label="Anomalías reales")
ax4a.set_xlabel("x₁")
ax4a.set_ylabel("x₂")
ax4a.set_title("Isolation Forest — detecciones")
ax4a.legend(fontsize=9)

# Scores
ax4b.hist(scores[etiquetas_reales == 0], bins=30, density=True,
          color="steelblue", alpha=0.6, edgecolor="white", label="Normales")
ax4b.hist(scores[etiquetas_reales == 1], bins=5, density=True,
          color="crimson", alpha=0.6, edgecolor="white", label="Anomalías reales")
ax4b.axvline(0, color="black", ls="--", lw=1.5, label="Umbral (score=0)")
ax4b.set_xlabel("Anomaly score")
ax4b.set_ylabel("Densidad")
ax4b.set_title("Distribución de scores")
ax4b.legend(fontsize=9)

plt.tight_layout()
plt.show()

> **Resultado:**
> - Las 5 anomalías inyectadas tienen scores negativos (más anómalos)
>   y son detectadas correctamente.
> - Los puntos normales tienen scores positivos (más "normales").
> - **En el laboratorio** aplicaremos Isolation Forest a múltiples
>   variables meteorológicas simultáneamente para captar anomalías
>   que los métodos univariados no detectan.

---
## Resumen

| Método | Idea clave | Fortaleza |
|:---|:---|:---|
| **STL** | Descomponer → buscar outliers en residuales | Robusto a outliers, maneja estacionalidad |
| **Umbral fijo** | ±3σ sobre residuales | Simple |
| **Percentiles** | P0.5/P99.5 | No asume normalidad |
| **Isolation Forest** | Aislar puntos con pocos cortes aleatorios | Multivariado, no paramétrico |

**Próxima sesión (10B):** Aplicar STL e Isolation Forest al dataset
ClimaLab y consolidar un DataFrame de flags de anomalías.